# Airbnb Paris – Experiment 1: AnoLLM
- LoRA-Finetuning Qwen2.5-0.5B auf serialisierten Zeilen, Score = NLL
- Unsupervised: Training auf dem **Train-Split ohne Labels**; lesbare Rohwerte + Freitexte
- Die Drop-Liste ist identisch zu `preprocessing/cleaned.ipynb`, damit beide Modellfamilien dieselben Informationen sehen

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import mlflow
import torch.distributed as dist
from torch.utils.data import DataLoader, SequentialSampler
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc

sys.path.insert(0, "../../anollm_src")
import anollm.anollm_trainer
from anollm.anollm import AnoLLM

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Rohdaten aufbereiten
- ICC-Filter + Label; Leakage-/ID-/Meta-Spalten droppen; lesbare Werte (%-Raten, Listen-Counts)

In [ ]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]

raw = pd.read_csv("../../data/raw/airbnb_paris.csv", low_memory=False)
raw = raw.dropna(subset=["review_scores_rating"])
raw = raw[(raw["review_scores_rating"] == 5.0) | (raw["review_scores_rating"] <= 3.0)].copy()
raw["row_id"] = raw["id"].astype("int64")
lab = pd.Series((raw["review_scores_rating"] != 5.0).astype(int).values, index=raw["row_id"].values)  # Outlier = 1

drop = ["id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url", "host_id", "host_url",
        "host_name", "host_thumbnail_url", "host_picture_url", "license",
        "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
        "review_scores_communication", "review_scores_location", "review_scores_value",
        "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
        "first_review", "last_review", "reviews_per_month", "calendar_updated", "calendar_last_scraped",
        "bathrooms_text", "host_listings_count", "host_total_listings_count",
        "minimum_minimum_nights", "maximum_minimum_nights", "minimum_maximum_nights", "maximum_maximum_nights",
        "minimum_nights_avg_ntm", "maximum_nights_avg_ntm", "has_availability", "host_neighbourhood",
        "neighbourhood", "price", "beds", "bathrooms", "estimated_revenue_l365d", "neighbourhood_group_cleansed"]
raw = raw.drop(columns=[c for c in drop if c in raw.columns])

raw["host_response_rate"] = pd.to_numeric(raw["host_response_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["host_acceptance_rate"] = pd.to_numeric(raw["host_acceptance_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["amenities_count"] = raw["amenities"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
raw["host_verifications_count"] = raw["host_verifications"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
feat = raw.drop(columns=["amenities", "host_verifications"]).set_index("row_id")

## Split laden & Imputation auf Train gefittet
- Dieselbe Split-Datei wie alle übrigen Modelle; der Median stammt ausschließlich aus Trainingszeilen

In [ ]:
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv")
train_ids = split.loc[split["split"] == "train", "row_id"].values
test_ids = split.loc[split["split"] == "test", "row_id"].values

num_cols = [c for c in feat.columns if pd.api.types.is_numeric_dtype(feat[c])]
str_cols = [c for c in feat.columns if c not in num_cols + text_cols]
feat[str_cols] = feat[str_cols].fillna("missing")
feat[text_cols] = feat[text_cols].fillna("")
feat[num_cols] = feat[num_cols].fillna(feat.loc[train_ids, num_cols].median())

df_train = feat.loc[train_ids].reset_index(drop=True)
df_test = feat.loc[test_ids].reset_index(drop=True)
y_train, y_test = lab.loc[train_ids].values, lab.loc[test_ids].values
k = round(len(y_test) * y_train.mean())
print("train", df_train.shape, "test", df_test.shape, "| Outlier-Rate Test:", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_1")

## Single-GPU-Setup (kein DDP/NCCL)
- Verteilte Env-Variablen entfernen und den Trainer-Dataloader patchen → kein DistributedDataParallel

In [ ]:
for key in ["LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(key, None)
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()

def single_gpu_get_train_dataloader(self):
    return DataLoader(self.train_dataset, batch_size=self._train_batch_size,
                      sampler=SequentialSampler(self.train_dataset),
                      collate_fn=self.data_collator, drop_last=True)

anollm.anollm_trainer.AnoLLMTrainer.get_train_dataloader = single_gpu_get_train_dataloader
print("Single-GPU-Modus aktiv, Trainer gepatcht.")

## AnoLLM trainieren & Scores (NLL)
- Gradient Checkpointing senkt den Trainingsspeicher (Airbnb hat längere Zeilen), das Modell bleibt identisch
- Schwelle für den Classification Report: Top-k Scores mit k = Testgröße × Outlier-Rate im Train

In [ ]:
model = AnoLLM(llm="Qwen/Qwen2.5-0.5B", efficient_finetuning="lora", textual_columns=text_cols,
               max_length_dict={c: 64 for c in text_cols}, batch_size=2, max_steps=1000, learning_rate=5e-4,
               gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
model.model.enable_input_require_grads()  # required for gradient checkpointing with LoRA/PEFT

t0 = time.perf_counter()
model.fit(df_train)
scores = model.decision_function(df_test, n_permutations=8, batch_size=2, device="cuda").mean(axis=1)
runtime = time.perf_counter() - t0

scores = np.asarray(scores).astype(float)
prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)
auroc = roc_auc_score(y_test, scores)
pred = (scores >= np.sort(scores)[-k]).astype(int)

with mlflow.start_run(run_name="anollm"):
    mlflow.log_params({"llm": "Qwen/Qwen2.5-0.5B", "max_steps": 1000, "n_permutations": 8, "seed": SEED})
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"anollm: AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} time={runtime:.1f}s")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))